# 50_error_analysis_and_reporting
**NOVA IMS – DSAA 2025/26**  
**Cars 4 You – Fehleranalyse, Modell-Diagnostik & Reporting**  
*Generated on:* 2025-10-17

Dieses Notebook baut auf dem 40er auf und liefert **robuste Diagnostik** für euer finales Regressionsmodell:
- OOF-Residualanalyse (Streuplots, Verteilungen, QQ-Check)
- Fehler-Slicing nach wichtigen Attributen (Top-K kateg./num. Gruppen)
- Permutation Importance (global)
- PDP/ICE für Top-Features
- Learning Curve (Bias/Variance-Einschätzung)
- Export von Abbildungen & kompaktem Markdown-Report

## Setup & Imports

In [ ]:
from pathlib import Path

DATA_DIR  = Path("data")
TRAIN_FILE = DATA_DIR / "train.csv"
TEST_FILE  = DATA_DIR / "test.csv"
TARGET = "price"

ARTIFACTS = Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True, parents=True)
DIAG_DIR = ARTIFACTS / "diagnostics"; DIAG_DIR.mkdir(exist_ok=True, parents=True)

print("DATA_DIR :", DATA_DIR)
print("TRAIN   :", TRAIN_FILE)
print("TEST    :", TEST_FILE)
print("TARGET  :", TARGET)
print("DIAG DIR:", DIAG_DIR.resolve())

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import KFold, learning_curve, cross_val_predict, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from scipy import stats

## Daten, Preprocessing & Featuremaske

In [ ]:
train = pd.read_csv(TRAIN_FILE)
test  = pd.read_csv(TEST_FILE)

assert TARGET in train.columns, f"Zielvariable '{TARGET}' nicht gefunden!"

X = train.drop(columns=[TARGET])
y = train[TARGET].copy()

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

numeric_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler(with_mean=True, with_std=True)),
])
categorical_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

SEL_FILE = ARTIFACTS / "selected_features.json"
assert SEL_FILE.exists(), f"Feature-Selektionsdatei nicht gefunden: {SEL_FILE.resolve()}"
meta = json.loads(SEL_FILE.read_text(encoding="utf-8"))
final_features = set(meta["final_features"])

_ = preprocessor.fit(X)

def get_feature_names(preprocessor, num_cols, cat_cols):
    feature_names = []
    feature_names += [f"NUM::{c}" for c in num_cols]
    ohe = preprocessor.named_transformers_["cat"].named_steps["ohe"]
    ohe_names = list(ohe.get_feature_names_out(cat_cols))
    feature_names += [f"CAT::{n}" for n in ohe_names]
    return feature_names

feature_names = get_feature_names(preprocessor, num_cols, cat_cols)
name_to_idx = {n:i for i, n in enumerate(feature_names)}
keep_idx = [name_to_idx[n] for n in feature_names if n in final_features]

def mask_final(X_):
    Xt = preprocessor.transform(X_)
    return Xt[:, keep_idx]

RANDOM_STATE = 42
CV_FOLDS = 10
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

scorer_mae  = make_scorer(mean_absolute_error, greater_is_better=False)

## Finales Modell re-fitten (aus 40)

In [ ]:
models_and_grids = {
    "LinearRegression": (LinearRegression(), {}),
    "Ridge": (Ridge(random_state=42), {"alpha": [0.1, 1.0, 3.0, 10.0, 30.0]}),
    "Lasso": (Lasso(random_state=42, max_iter=20000), {"alpha": [0.0005, 0.001, 0.01, 0.1, 1.0]}),
    "ElasticNet": (ElasticNet(random_state=42, max_iter=20000), {
        "alpha": [0.0005, 0.001, 0.01, 0.1, 1.0],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
    }),
    "RandomForest": (RandomForestRegressor(random_state=42, n_jobs=-1), {
        "n_estimators": [600],
        "max_depth": [None, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }),
    "GradientBoosting": (GradientBoostingRegressor(random_state=42), {
        "n_estimators": [600],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3],
        "subsample": [0.8, 1.0]
    }),
    "SVR": (SVR(), {
        "C": [3.0, 10.0],
        "epsilon": [0.01, 0.05],
        "kernel": ["rbf"],
        "gamma": ["scale"]
    }),
}

BEST_MODEL_NAME = "RandomForest"  # <<< ggf. aus 40er übernehmen

estimator, grid = models_and_grids[BEST_MODEL_NAME]

pipe = Pipeline([
    ("mask", FunctionTransformer(lambda Z: mask_final(Z), feature_names_out="one-to-one")),
    ("model", estimator)
])

if grid:
    gs = GridSearchCV(pipe, param_grid={f"model__{k}": v for k, v in grid.items()},
                      scoring=scorer_mae, cv=5, n_jobs=-1, refit=True)
    gs.fit(X, y)
    final_model = gs.best_estimator_
    best_params = gs.best_params_
else:
    final_model = pipe.fit(X, y)
    best_params = {}

print("Final model:", BEST_MODEL_NAME, "| Best params:", best_params)

## OOF-Residualanalyse

In [ ]:
y_oof = cross_val_predict(final_model, X, y, cv=kf, n_jobs=-1)
residuals = y - y_oof

oof_mae  = mean_absolute_error(y, y_oof)
oof_rmse = np.sqrt(mean_squared_error(y, y_oof))
oof_r2   = r2_score(y, y_oof)
print(f"OOF MAE: {oof_mae:.4f} | OOF RMSE: {oof_rmse:.4f} | OOF R²: {oof_r2:.4f}")

plt.figure()
plt.scatter(y_oof, residuals, s=8, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted (OOF)")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted (OOF)")
plt.tight_layout()
plt.savefig(DIAG_DIR / "residuals_vs_pred.png", dpi=160)
plt.show()

plt.figure()
plt.hist(residuals, bins=40)
plt.title("Residuals Distribution")
plt.tight_layout()
plt.savefig(DIAG_DIR / "residuals_hist.png", dpi=160)
plt.show()

z = (residuals - residuals.mean()) / residuals.std(ddof=1)
_ = stats.probplot(z, dist="norm", plot=plt)
plt.tight_layout()
plt.savefig(DIAG_DIR / "residuals_qq.png", dpi=160)
plt.show()

## Fehler-Slicing (kategorial & numerisch)

In [ ]:
df = X.copy()
df[TARGET] = y
df["yhat_oof"] = y_oof
df["abs_err"] = (df[TARGET] - df["yhat_oof"]).abs()

def topk_categories(col, k=8):
    return df[col].value_counts().nlargest(k).index.tolist()

slice_reports = []

for c in cat_cols[:6]:
    cats = topk_categories(c, k=8)
    tmp = (df[df[c].isin(cats)]
           .groupby(c)["abs_err"]
           .agg(["count", "mean", "median", "std"]
           ).sort_values("mean", ascending=False).rename_axis(c))
    tmp["feature"] = c
    slice_reports.append(tmp.reset_index())

for c in num_cols[:6]:
    try:
        q = pd.qcut(df[c], q=10, duplicates="drop")
        tmp = (df.groupby(q)["abs_err"]
               .agg(["count", "mean", "median", "std"]
               ).sort_values("mean", ascending=False))
        tmp["feature"] = c
        slice_reports.append(tmp.reset_index().rename(columns={c: f"{c}_deciles"}))
    except Exception:
        pass

slices = pd.concat(slice_reports, axis=0, ignore_index=True)
slices.head(20)

## Globale Feature-Bedeutung (Permutation Importance)

In [ ]:
final_model.fit(X, y)
perm = permutation_importance(final_model, X, y, n_repeats=10, random_state=42, n_jobs=-1)

importances = pd.DataFrame({
    "feature": [f for f in feature_names if f in final_features],
    "importance": perm.importances_mean[:len(final_features)]
}).sort_values("importance", ascending=False)

top10 = importances.head(10)

plt.figure()
plt.barh(top10["feature"][::-1], top10["importance"][::-1])
plt.title("Permutation Importance (Top 10)")
plt.tight_layout()
plt.savefig(DIAG_DIR / "perm_importance_top10.png", dpi=160)
plt.show()

importances.head(20)

## PDP/ICE für Top-Features

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class SelectColsByIndex(BaseEstimator, TransformerMixin):
    def __init__(self, indices):
        self.indices = indices
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        Xt = preprocessor.transform(X)
        return Xt[:, self.indices]

top_feats = importances["feature"].head(4).tolist()
top_idx = [name_to_idx[t] for t in top_feats]

model_only = final_model.named_steps.get("model", final_model)

pipe_for_pdp = Pipeline([
    ("prep_select", SelectColsByIndex(keep_idx)),
    ("model", model_only)
]).fit(X, y)

for feat in top_feats:
    feat_idx = [name_to_idx[feat]]
    try:
        disp = PartialDependenceDisplay.from_estimator(
            pipe_for_pdp, X, features=[feat_idx], kind="both"
        )
        plt.tight_layout()
        safe_name = feat.replace("/", "_").replace(":", "_").replace(" ", "_")
        plt.savefig(DIAG_DIR / f"pdp_ice_{safe_name}.png", dpi=160)
        plt.show()
    except Exception as e:
        print("PDP failed for", feat, "->", e)

## Learning Curve

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    final_model, X, y, cv=5, scoring="neg_mean_absolute_error",
    train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=-1, shuffle=True, random_state=42
)

train_mae = -train_scores.mean(axis=1)
val_mae   = -val_scores.mean(axis=1)

plt.figure()
plt.plot(train_sizes, train_mae, marker="o", label="Train MAE")
plt.plot(train_sizes, val_mae, marker="o", label="CV MAE")
plt.xlabel("Train size")
plt.ylabel("MAE")
plt.title("Learning Curve (MAE)")
plt.legend()
plt.tight_layout()
plt.savefig(DIAG_DIR / "learning_curve_mae.png", dpi=160)
plt.show()

## Report Export

In [ ]:
report_md = f"""
# Cars4You – Fehleranalyse & Modell-Diagnostik
*Erstellt am:* {datetime.now().strftime("%Y-%m-%d")}

## Zusammenfassung
- OOF MAE: {oof_mae:.4f} | OOF RMSE: {oof_rmse:.4f} | OOF R²: {oof_r2:.4f}
- Finales Modell: {BEST_MODEL_NAME} | Beste Params: {best_params}

## Wichtigste Artefakte
- Residualplots: `residuals_vs_pred.png`, `residuals_hist.png`, `residuals_qq.png`
- Permutation Importance (Top 10): `perm_importance_top10.png`
- PDP/ICE (Top 4 Features): `pdp_ice_*.png`
- Learning Curve: `learning_curve_mae.png`

## Fehler-Slicing (Top-Auszüge)
Nach Kategorie/Numerik gruppierte Fehlerstatistik liegt in `slices` (siehe Notebook-Zelle "Error Slicing").
"""

out_report = DIAG_DIR / "report_error_analysis.md"
out_report.write_text(report_md, encoding="utf-8")
print("Report gespeichert:", out_report.resolve())